# Estudo guiado — Capítulo 4: construindo um GPT

Este notebook acompanha o estudo do Capítulo 4 e transforma os exercícios 4.1, 4.2 e 4.3 em uma atividade guiada. Vamos comparar parâmetros, estimar o tamanho de diferentes configurações GPT-2 e implementar uma versão pequena com taxas de dropout separadas.

## Objetivos

- Entender como `FeedForward` e atenção contribuem para os parâmetros de um bloco Transformer.
- Comparar GPT-2 small, medium, large e XL sem alocar modelos gigantes na memória.
- Adaptar o bloco e o modelo para usar dropout independente nos embeddings, na atenção e nas conexões residuais.
- Conferir dimensões e propriedades com testes executáveis.

> Este capítulo monta a arquitetura e demonstra geração com pesos aleatórios. **Não treina o GPT nem avalia a qualidade da linguagem gerada.** Execute as células em ordem. A atividade usa a implementação oficial do repositório para componentes já estudados.

## 1. Importação de bibliotecas

Usaremos PyTorch para contar parâmetros e executar uma configuração pequena. `pathlib` e `sys` ajudam a localizar os módulos do capítulo sem depender de uma pasta de execução específica. Não é necessário instalar nada automaticamente: o ambiente do repositório já deve ter PyTorch.

In [12]:
from pathlib import Path
import ast

import torch
from torch import nn

torch.manual_seed(123)

CODE_DIR = None
for base in (Path.cwd(), *Path.cwd().parents):
    for relative_path in (
        Path("ch04/01_main-chapter-code"),
        Path("01_main-chapter-code"),
    ):
        candidate = base / relative_path
        if (candidate / "gpt.py").is_file():
            CODE_DIR = candidate.resolve()
            break
    if CODE_DIR is not None:
        break

if CODE_DIR is None:
    raise FileNotFoundError(
        "Não encontrei ch04/01_main-chapter-code/gpt.py. "
        "Abra este notebook dentro do repositório LLMs-from-scratch."
    )

# Carrega do arquivo oficial apenas as classes necessárias; o tokenizer
# tiktoken não é usado nesta atividade e, portanto, não precisa ser instalado.
source_path = CODE_DIR / "gpt.py"
source_tree = ast.parse(source_path.read_text(encoding="utf-8"))
classes_necessarias = {
    "MultiHeadAttention", "LayerNorm", "GELU", "FeedForward", "TransformerBlock", "GPTModel"
}
class_nodes = [
    node for node in source_tree.body
    if isinstance(node, ast.ClassDef) and node.name in classes_necessarias
]
namespace = {"torch": torch, "nn": nn}
exec(compile(ast.Module(body=class_nodes, type_ignores=[]), str(source_path), "exec"), namespace)

FeedForward = namespace["FeedForward"]
LayerNorm = namespace["LayerNorm"]
MultiHeadAttention = namespace["MultiHeadAttention"]
TransformerBlockOficial = namespace["TransformerBlock"]

print("PyTorch:", torch.__version__)
print("Componentes carregados de:", source_path)
print("Dispositivo desta atividade: CPU")

PyTorch: 2.14.1+cpu
Componentes carregados de: /home/danielanderson.silva.copin/Downloads/transformer-model-engineering
Public/LLMs-from-scratch/ch04/01_main-chapter-code/gpt.py
Dispositivo desta atividade: CPU


## 2. Leitura do texto de referência

Vamos carregar o README que acompanha o código oficial do capítulo. Em seguida, use o resumo abaixo como texto-base: o GPT soma embeddings de tokens e posições; cada bloco aplica atenção causal e uma rede feed-forward com conexões residuais; a saída contém logits para prever o próximo token. Os exercícios investigam os parâmetros e a configuração desses componentes.

In [13]:
readme_path = CODE_DIR / "README.md"
texto_referencia = readme_path.read_text(encoding="utf-8")

print("Arquivo lido:", readme_path)
print("\nTrecho de referência:\n")
print(texto_referencia[:1200])

# Este resumo será reutilizado na explicação final gerada pelo notebook.
resumo_capitulo = (
    "O Capítulo 4 implementa a arquitetura GPT: embeddings de tokens e posições, "
    "blocos Transformer com atenção causal e feed-forward, normalização e uma "
    "camada de saída que produz logits para prever o próximo token."
)
print("\nTexto-base da atividade:\n", resumo_capitulo)

Arquivo lido: /home/danielanderson.silva.copin/Downloads/transformer-model-engineering
Public/LLMs-from-scratch/ch04/01_main-chapter-code/README.md

Trecho de referência:

# Chapter 4: Implementing a GPT Model from Scratch To Generate Text

### Main Chapter Code

- [ch04.ipynb](ch04.ipynb) contains all the code as it appears in the chapter
- [previous_chapters.py](previous_chapters.py) is a Python module that contains the `MultiHeadAttention` module from the previous chapter, which we import in [ch04.ipynb](ch04.ipynb) to create the GPT model

### Optional Code

- [gpt.py](gpt.py) is a standalone Python script file with the code that we implemented thus far, including the GPT model we coded in this chapter



Texto-base da atividade:
 O Capítulo 4 implementa a arquitetura GPT: embeddings de tokens e posições, blocos Transformer com atenção causal e feed-forward, normalização e uma camada de saída que produz logits para prever o próximo token.


## 3. Execução da atividade relacionada ao texto

### Exercício 4.1 — parâmetros: feed-forward versus atenção

Um bloco contém uma `MultiHeadAttention` e uma `FeedForward`. Vamos contar os parâmetros aprendíveis diretamente nos módulos oficiais. Para dimensão $d=768$, a camada feed-forward expande para $4d$ e volta a $d$; a atenção usa projeções Q, K, V e uma projeção de saída. As projeções QKV estão sem bias nesta configuração, conforme o capítulo.

In [14]:
config_124m = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

bloco_oficial = TransformerBlockOficial(config_124m)
parametros_ffn = sum(p.numel() for p in bloco_oficial.ff.parameters())
parametros_atencao = sum(p.numel() for p in bloco_oficial.att.parameters())

print(f"Feed-forward em um bloco: {parametros_ffn:,} parâmetros")
print(f"Atenção em um bloco:      {parametros_atencao:,} parâmetros")
print(f"Razão feed-forward/atenção: {parametros_ffn / parametros_atencao:.3f}x")

# Fórmulas para comparar a contagem do módulo com a conta manual.
d = config_124m["emb_dim"]
ffn_manual = (d * 4 * d + 4 * d) + (4 * d * d + d)
atencao_manual = 3 * d * d + (d * d + d)  # Q, K, V sem bias; out_proj com bias

assert parametros_ffn == ffn_manual
assert parametros_atencao == atencao_manual
print("\nContagens conferidas pelas fórmulas ✅")

Feed-forward em um bloco: 4,722,432 parâmetros
Atenção em um bloco:      2,360,064 parâmetros
Razão feed-forward/atenção: 2.001x

Contagens conferidas pelas fórmulas ✅


A `FeedForward` tem aproximadamente o dobro dos parâmetros da atenção em um único bloco. Isso ocorre porque ela faz duas projeções grandes, $d\to4d$ e $4d\to d$. A atenção faz quatro projeções de largura $d$ (três de entrada e uma de saída). Essa comparação é por bloco; o modelo repete os blocos várias vezes.

### Exercício 4.2 — configurações GPT-2 maiores

As configurações variam a dimensão de embedding, o número de blocos e o número de cabeças. Construir GPT-2 XL na memória pode exigir vários gigabytes, então vamos calcular os parâmetros pela arquitetura, sem alocar as matrizes. Para este modelo, a cabeça de saída tem pesos próprios (não há compartilhamento de pesos, ou *weight tying*).

In [15]:
TAMANHO_VOCAB = 50257
CONTEXTO = 1024
TAXA_DROPOUT = 0.1

modelos = {
    "GPT-2 small": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "GPT-2 medium": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "GPT-2 large": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "GPT-2 XL": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

def contar_parametros_gpt(config):
    """Conta parâmetros da arquitetura do capítulo, sem criar matrizes."""
    d = config["emb_dim"]
    n_layers = config["n_layers"]
    vocab = config["vocab_size"]
    context = config["context_length"]

    # Embedding de tokens e cabeça de saída não compartilham pesos.
    embeddings = vocab * d + context * d
    output_head = vocab * d

    # Por bloco: atenção (4d² + d), FFN (8d² + 5d), LayerNorms (4d).
    parametros_por_bloco = 12 * d**2 + 10 * d
    blocos = n_layers * parametros_por_bloco
    norm_final = 2 * d
    return embeddings + output_head + blocos + norm_final

resultados_modelos = []
for nome, dimensoes in modelos.items():
    cfg = {
        **dimensoes,
        "vocab_size": TAMANHO_VOCAB,
        "context_length": CONTEXTO,
        "drop_rate": TAXA_DROPOUT,
        "qkv_bias": False,
    }
    n_params = contar_parametros_gpt(cfg)
    memoria_mib = n_params * 4 / (1024**2)  # float32: 4 bytes por parâmetro
    resultados_modelos.append((nome, cfg, n_params, memoria_mib))

print(f"{'Modelo':<14} {'d_model':>8} {'blocos':>8} {'cabeças':>8} {'parâmetros':>16} {'float32 MiB':>14}")
for nome, cfg, n_params, memoria_mib in resultados_modelos:
    print(f"{nome:<14} {cfg['emb_dim']:>8} {cfg['n_layers']:>8} {cfg['n_heads']:>8} {n_params:>16,} {memoria_mib:>14,.1f}")

# A contagem da configuração small deve reproduzir a implementação não amarrada do capítulo.
assert resultados_modelos[0][2] == 163_009_536
print("\nContagem GPT-2 small conferida: 163.009.536 parâmetros ✅")

Modelo          d_model   blocos  cabeças       parâmetros    float32 MiB
GPT-2 small         768       12       12      163,009,536          621.8
GPT-2 medium       1024       24       16      406,212,608        1,549.6
GPT-2 large        1280       36       20      838,220,800        3,197.6
GPT-2 XL           1600       48       25    1,637,792,000        6,247.7

Contagem GPT-2 small conferida: 163.009.536 parâmetros ✅


A contagem considera pesos e biases de todas as camadas, embeddings de tokens e posições, blocos Transformer, normalização final e a cabeça de saída. A memória é uma estimativa **apenas dos pesos em float32** ($4$ bytes por parâmetro); durante o treinamento, gradientes e estados do otimizador aumentariam bastante o consumo. As estimativas evitam alocar os modelos medium/large/XL e, assim, reduzem o risco de falta de memória.

### Exercício 4.3 — separar as taxas de dropout

No modelo básico, a mesma taxa controla dropout nos embeddings, nos pesos de atenção e nas saídas que passam pelas conexões residuais. Vamos tornar esses controles independentes. A atenção, a normalização e a rede feed-forward continuam sendo os componentes oficiais; adaptamos somente onde cada taxa é aplicada.

In [10]:
class TransformerBlockDropoutSeparado(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate_attn"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate_shortcut"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModelDropoutSeparado(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate_emb"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlockDropoutSeparado(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        if seq_len > self.pos_emb.num_embeddings:
            raise ValueError("A sequência excede o context_length configurado.")
        token_embeddings = self.tok_emb(in_idx)
        position_ids = torch.arange(seq_len, device=in_idx.device)
        position_embeddings = self.pos_emb(position_ids)
        x = self.drop_emb(token_embeddings + position_embeddings)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


config_tiny = {
    "vocab_size": 97,
    "context_length": 8,
    "emb_dim": 32,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate_emb": 0.1,
    "drop_rate_attn": 0.2,
    "drop_rate_shortcut": 0.3,
    "qkv_bias": False,
}

modelo_tiny = GPTModelDropoutSeparado(config_tiny)
entrada = torch.randint(0, config_tiny["vocab_size"], (2, 5))  # (lote=2, tokens=5)
logits_tiny = modelo_tiny(entrada)

print("Entrada:", tuple(entrada.shape))
print("Logits:", tuple(logits_tiny.shape), "= (lote, tokens, vocabulário)")
print("Dropout embeddings:", modelo_tiny.drop_emb.p)
print("Dropout atenção:", modelo_tiny.trf_blocks[0].att.dropout.p)
print("Dropout atalhos:", modelo_tiny.trf_blocks[0].drop_shortcut.p)

assert logits_tiny.shape == (2, 5, config_tiny["vocab_size"])
assert modelo_tiny.drop_emb.p == config_tiny["drop_rate_emb"]
assert modelo_tiny.trf_blocks[0].att.dropout.p == config_tiny["drop_rate_attn"]
assert modelo_tiny.trf_blocks[0].drop_shortcut.p == config_tiny["drop_rate_shortcut"]
print("Dimensões e taxas de dropout conferidas ✅")

Entrada: (2, 5)
Logits: (2, 5, 97) = (lote, tokens, vocabulário)
Dropout embeddings: 0.1
Dropout atenção: 0.2
Dropout atalhos: 0.3
Dimensões e taxas de dropout conferidas ✅


### Verificação de aprendizado

Antes de encerrar, reflita:

- Por que a `FeedForward` tem mais parâmetros que a atenção em um bloco?
- Por que estimamos o tamanho de GPT-2 XL sem instanciá-lo?
- Onde cada taxa de dropout independente é aplicada?
- O que a forma dos logits nos diz — e o que ela **não** diz sobre a qualidade do modelo?

Os asserts anteriores verificam a aritmética dos parâmetros e o fluxo de tensores. Eles não representam treinamento: para aprender a prever texto, ainda seriam necessários dados, alvos, função de perda, otimizador e laço de treinamento.

## 4. Geração de um novo texto explicativo da atividade

A última célula monta um texto a partir dos resultados calculados acima. Assim, ao alterar dimensões ou taxas no notebook, a explicação usa os valores que foram realmente obtidos.

In [16]:
texto_explicativo = f"""
Nesta atividade, lemos a referência do Capítulo 4 e trabalhamos três aspectos da arquitetura GPT.

No exercício 4.1, contamos os parâmetros de um bloco oficial: a rede feed-forward possui {parametros_ffn:,} parâmetros e a atenção possui {parametros_atencao:,}. Portanto, a feed-forward tem cerca de {parametros_ffn / parametros_atencao:.2f} vezes os parâmetros da atenção, principalmente por expandir a dimensão interna para quatro vezes o tamanho do embedding.

No exercício 4.2, comparamos as quatro configurações GPT-2 por uma fórmula, sem alocar seus pesos. A configuração small tem {resultados_modelos[0][2]:,} parâmetros nesta implementação sem compartilhamento entre embeddings e cabeça de saída. A memória estimada para os pesos dessa configuração em float32 é {resultados_modelos[0][3]:,.1f} MiB; configurações maiores exigem muito mais memória, e o treinamento acrescentaria gradientes e estados do otimizador.

No exercício 4.3, implementamos um modelo pequeno que separa dropout de embeddings ({config_tiny['drop_rate_emb']:.0%}), atenção ({config_tiny['drop_rate_attn']:.0%}) e conexões residuais ({config_tiny['drop_rate_shortcut']:.0%}). Para uma entrada com forma {tuple(entrada.shape)}, a saída tem forma {tuple(logits_tiny.shape)}: há uma distribuição de logits sobre o vocabulário em cada posição de cada sequência.

Esses resultados verificam contagens, dimensões e a configuração dos componentes. Eles não mostram que o modelo aprendeu linguagem: os pesos não foram treinados nesta atividade.
""".strip()

print(texto_explicativo)

Nesta atividade, lemos a referência do Capítulo 4 e trabalhamos três aspectos da arquitetura GPT.

No exercício 4.1, contamos os parâmetros de um bloco oficial: a rede feed-forward possui 4,722,432 parâmetros e a atenção possui 2,360,064. Portanto, a feed-forward tem cerca de 2.00 vezes os parâmetros da atenção, principalmente por expandir a dimensão interna para quatro vezes o tamanho do embedding.

No exercício 4.2, comparamos as quatro configurações GPT-2 por uma fórmula, sem alocar seus pesos. A configuração small tem 163,009,536 parâmetros nesta implementação sem compartilhamento entre embeddings e cabeça de saída. A memória estimada para os pesos dessa configuração em float32 é 621.8 MiB; configurações maiores exigem muito mais memória, e o treinamento acrescentaria gradientes e estados do otimizador.

No exercício 4.3, implementamos um modelo pequeno que separa dropout de embeddings (10%), atenção (20%) e conexões residuais (30%). Para uma entrada com forma (2, 5), a saída tem f